##  Carga de las fuentes de información

In [2]:
# Instala las librerías a usar
!pip install -q gdown
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q langchain-text-splitters
!pip install -q "grafitodb[viz]" matplotlib
!pip install -q transformers
!pip install -q accelerate
!pip install -q txtai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [3]:
import gdown
import zipfile
import os
import pandas as pd
import json
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import CharacterTextSplitter
from grafito import GrafitoDatabase
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from txtai.scoring import ScoringFactory
from sentence_transformers import util

###  Descarga del dataset desde Google Drive

In [4]:
file_id = "1LY9FWZzuB-KSrdWkHAMDtNk2LemWLxls" #ID del archivo zip en google
output = "fuentes_de_informacion.zip"

gdown.download(id=file_id, output=output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1LY9FWZzuB-KSrdWkHAMDtNk2LemWLxls
To: /content/fuentes_de_informacion.zip
100%|██████████| 3.81M/3.81M [00:00<00:00, 83.1MB/s]


'fuentes_de_informacion.zip'

### Descompresión del archivo ZIP

In [5]:
# Carpeta donde vamos a descomprimir todo
extract_path = "/content/fuentes_de_informacion"

# Crea la carpeta si no existe
os.makedirs(extract_path, exist_ok=True)

# Abro el ZIP y extraigo todo adentro de extract_path
with zipfile.ZipFile(output, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP descomprimido")

ZIP descomprimido


In [6]:
# Ruta base real del dataset.
# Quedó una carpeta fuentes_de_informacion adentro de otra por cómo estaba armado el ZIP.
base_path = Path("/content/fuentes_de_informacion/fuentes_de_informacion")

# Rutas a las carpetas de textos
resenas_path = base_path / "resenas_usuarios"
manuales_path = base_path / "manuales_productos"

# Rutas a archivos principales
productos_csv_path = base_path / "productos.csv"
productos_xlsx_path = base_path / "productos.xlsx"
inventario_path = base_path / "inventario_sucursales.csv"
ventas_path = base_path / "ventas_historicas.csv"
devoluciones_path = base_path / "devoluciones.csv"
tickets_path = base_path / "tickets_soporte.csv"
vendedores_path = base_path / "vendedores.csv"
faqs_path = base_path / "faqs.json"

# Verificamos rápido que las rutas existan
print("Base:", base_path.exists())
print("Reseñas:", resenas_path.exists())
print("Manuales:", manuales_path.exists())
print("Productos CSV:", productos_csv_path.exists())
print("FAQs:", faqs_path.exists())

Base: True
Reseñas: True
Manuales: True
Productos CSV: True
FAQs: True


### Carga de archivos tabulares

In [7]:
productos_df = pd.read_csv(productos_csv_path)
inventario_df = pd.read_csv(inventario_path)
ventas_df = pd.read_csv(ventas_path)
devoluciones_df = pd.read_csv(devoluciones_path)
tickets_df = pd.read_csv(tickets_path)
vendedores_df = pd.read_csv(vendedores_path)

## Modelo de lenguaje local auxiliar

In [8]:
# Se usa un modelo local liviano para tareas de generación estructurada.
# Más adelante se va a usar para generar filtros tabulares y consultas Cypher.

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer_llm = AutoTokenizer.from_pretrained(MODEL_ID)

model_llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Modelo local cargado:", MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo local cargado: Qwen/Qwen2.5-3B-Instruct


In [9]:
# Función auxiliar para consultar el modelo local.

def consultar_llm_local(system_prompt, user_prompt, max_new_tokens=150):
    mensajes = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt = tokenizer_llm.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_llm(
        prompt,
        return_tensors="pt"
    ).to(model_llm.device)

    salida = model_llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer_llm.eos_token_id
    )

    texto_generado = tokenizer_llm.decode(
        salida[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return texto_generado.strip()

##  Diseño de las fuentes de datos

El sistema utilizará tres fuentes de conocimiento con propósitos distintos:

- Base vectorial: para recuperar información textual no estructurada mediante similitud semántica.
- Base tabular: para responder consultas que requieren filtros, comparaciones, rangos o agregaciones.
- Base de grafos: para representar relaciones entre productos, categorías, subcategorías y marcas.

Esta separación va a permitir elegir la fuente más adecuada según la intención de la consulta del usuario.

In [10]:
# Resumen de diseño de las fuentes que vamos a usar en el sistema

disenio_fuentes = pd.DataFrame([
    {
        "fuente": "Base vectorial",
        "motor": "ChromaDB",
        "informacion": "Manuales, FAQs, reseñas y descripciones textuales de tickets de soporte",
        "uso": "Preguntas sobre uso de productos, opiniones, problemas frecuentes y respuestas en lenguaje natural"
    },
    {
        "fuente": "Base tabular",
        "motor": "Pandas",
        "informacion": "Productos, inventario, ventas, devoluciones, tickets y vendedores",
        "uso": "Consultas con filtros, precios, stock, categorías, ventas, devoluciones y métricas"
    },
    {
        "fuente": "Base de grafos",
        "motor": "GrafitoDB",
        "informacion": "Relaciones entre productos, categorías, subcategorías y marcas",
        "uso": "Consultas sobre relaciones, productos conectados y navegación por categorías"
    }
])

display(disenio_fuentes)

,fuente,motor,informacion,uso
0,Base vectorial,ChromaDB,"Manuales, FAQs, reseñas y descripciones textua...","Preguntas sobre uso de productos, opiniones, p..."
1,Base tabular,Pandas,"Productos, inventario, ventas, devoluciones, t...","Consultas con filtros, precios, stock, categor..."
2,Base de grafos,GrafitoDB,"Relaciones entre productos, categorías, subcat...","Consultas sobre relaciones, productos conectad..."


La base vectorial se utiliza para textos no estructurados, como manuales, reseñas, FAQs y tickets, porque permite recuperar fragmentos por similitud semántica.

La base tabular se utiliza para datos estructurados, como precios, stock, ventas, devoluciones e inventario, porque permite aplicar filtros exactos y rangos numéricos.

La base de grafos se utiliza para representar relaciones entre productos, categorías, subcategorías y marcas, porque permite consultar conexiones mediante Cypher.

##  Preparación de documentos para la base vectorial

In [11]:
documentos_vectoriales = []

# 1) FAQs
# Las FAQs sirven para preguntas frecuentes directas de usuarios.
with open(faqs_path, "r", encoding="utf-8") as f:
    faqs_data = json.load(f)

for i, faq in enumerate(faqs_data):
    texto = " ".join([str(v) for v in faq.values()])

    documentos_vectoriales.append({
        "id": f"faq_{i}",
        "texto": texto,
        "metadata": {
            "fuente": "faqs",
            "tipo": "pregunta_frecuente"
        }
    })

# 2) Manuales de productos
# Los manuales sirven para consultas sobre uso, mantenimiento y especificaciones.
for archivo in manuales_path.glob("*.md"):
    with open(archivo, "r", encoding="utf-8") as f:
        texto = f.read()

    documentos_vectoriales.append({
        "id": f"manual_{archivo.stem}",
        "texto": texto,
        "metadata": {
            "fuente": "manuales_productos",
            "tipo": "manual",
            "archivo": archivo.name
        }
    })

# 3) Reseñas de usuarios
# Las reseñas sirven para preguntas sobre opiniones y experiencia de usuarios.
for archivo in resenas_path.glob("*.txt"):
    with open(archivo, "r", encoding="utf-8") as f:
        texto = f.read()

    documentos_vectoriales.append({
        "id": f"resena_{archivo.stem}",
        "texto": texto,
        "metadata": {
            "fuente": "resenas_usuarios",
            "tipo": "resena",
            "archivo": archivo.name
        }
    })

# 4) Tickets de soporte
# Usamos la descripción textual de los tickets para recuperar problemas similares.
for _, row in tickets_df.iterrows():
    texto = f"""
    Producto: {row['nombre_producto']}
    Tipo de problema: {row['tipo_problema']}
    Descripción: {row['descripcion']}
    Severidad: {row['severidad']}
    Categoría: {row['categoria']}
    Estado: {row['estado']}
    Garantía válida: {row['garantia_valida']}
    """

    documentos_vectoriales.append({
        "id": f"ticket_{row['id_ticket']}",
        "texto": texto,
        "metadata": {
            "fuente": "tickets_soporte",
            "tipo": "ticket",
            "id_producto": row["id_producto"],
            "nombre_producto": row["nombre_producto"],
            "categoria": row["categoria"]
        }
    })
print("Documentos preparados para ChromaDB:", len(documentos_vectoriales))


Documentos preparados para ChromaDB: 10065


## Base de datos vectorial con ChromaDB

In [12]:
# Modelo multilingüe apto para español.
# Es chico, rápido y sirve para búsqueda semántica.
embedding_model_name = "intfloat/multilingual-e5-small"

embedding_model = SentenceTransformer(embedding_model_name)

print("Modelo cargado:", embedding_model_name)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Modelo cargado: intfloat/multilingual-e5-small


###  Segmentación de documentos

In [13]:
# Dividimos los textos largos en fragmentos más chicos.
# Esto ayuda a que ChromaDB recupere partes más puntuales y no documentos enormes.

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=700,
    chunk_overlap=100
)

fragmentos_vectoriales = []

for doc in documentos_vectoriales:
    partes = text_splitter.split_text(doc["texto"])

    for i, parte in enumerate(partes):
        fragmentos_vectoriales.append({
            "id": f"{doc['id']}_chunk_{i}",
            "texto": parte,
            "metadata": {
                **doc["metadata"],
                "doc_id": doc["id"],
                "chunk": i
            }
        })

print("Documentos originales:", len(documentos_vectoriales))
print("Fragmentos generados:", len(fragmentos_vectoriales))

Documentos originales: 10065
Fragmentos generados: 10575


### Creación de la colección en ChromaDB

In [14]:
# Creamos el cliente local de ChromaDB.
# El cliente es el objeto que administra las colecciones.
client = chromadb.Client()

# Nombre de la colección donde vamos a guardar los fragmentos.
collection_name = "electrodomesticos_docs"

# Si la colección ya existía de una ejecución anterior, la borra.
# Esto evita cargar documentos duplicados si volvemos a correr la celda.
try:
    client.delete_collection(name=collection_name)
except:
    pass

# Creamos la colección nueva.
collection = client.create_collection(name=collection_name)

print("Colección creada:", collection_name)

Colección creada: electrodomesticos_docs


### Generación de embeddings

In [15]:
# Separamos la información en listas porque ChromaDB trabaja con listas paralelas:
# ids: identificadores únicos
# documents: textos
# metadatas: información extra de cada fragmento

ids = [frag["id"] for frag in fragmentos_vectoriales]
documents = [frag["texto"] for frag in fragmentos_vectoriales]
metadatas = [frag["metadata"] for frag in fragmentos_vectoriales]

# Como usamos el modelo E5, agregamos "passage:" delante de los documentos.
# Esto ayuda al modelo a entender que estos textos son pasajes a recuperar.
documents_for_embedding = ["passage: " + texto for texto in documents]

# Generamos los embeddings.
# batch_size indica cuántos textos procesa juntos.
# normalize_embeddings=True deja los vectores normalizados para comparación semántica.
embeddings = embedding_model.encode(
    documents_for_embedding,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).tolist()

print("Embeddings generados:", len(embeddings))

Batches:   0%|          | 0/166 [00:00<?, ?it/s]

Embeddings generados: 10575


### Carga de fragmentos en ChromaDB

In [16]:
# Cargo los fragmentos en ChromaDB.
# Lo hago en tandas para no mandar todo junto y evitar problemas de memoria.

batch_size = 1000

for i in range(0, len(documents), batch_size):

    collection.add(
        ids=ids[i:i+batch_size],
        documents=documents[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size]
    )

    print(f"Cargados {min(i + batch_size, len(documents))} de {len(documents)}")

print("Carga finalizada.")

Cargados 1000 de 10575
Cargados 2000 de 10575
Cargados 3000 de 10575
Cargados 4000 de 10575
Cargados 5000 de 10575
Cargados 6000 de 10575
Cargados 7000 de 10575
Cargados 8000 de 10575
Cargados 9000 de 10575
Cargados 10000 de 10575
Cargados 10575 de 10575
Carga finalizada.


In [17]:
# Verificación de carga en ChromaDB
print("Cantidad de fragmentos en ChromaDB:", collection.count())

Cantidad de fragmentos en ChromaDB: 10575


### Interfaz de búsqueda en ChromaDB

In [18]:
def buscar_en_chroma(consulta, k=5, filtros=None):
    # Como uso el modelo E5, la consulta lleva el prefijo "query:"
    # Esto le indica al modelo que este texto es una pregunta del usuario.
    consulta_embedding = embedding_model.encode(
        ["query: " + consulta],
        normalize_embeddings=True
    ).tolist()

    # Busco en la colección de ChromaDB.
    # n_results=k indica cuántos fragmentos quiero recuperar.
    # where=filtros permite limitar la búsqueda por metadata.
    resultados = collection.query(
        query_embeddings=consulta_embedding,
        n_results=k,
        where=filtros
    )

    return resultados

### Resumen de la base tabular

In [19]:
# Se arma un resumen de la tabla de productos.
# La idea no es pasarle todos los productos al modelo, sino solamente información útil:
# columnas disponibles, valores posibles y rangos numéricos.

categorias = sorted(productos_df["categoria"].dropna().unique().tolist())
subcategorias = sorted(productos_df["subcategoria"].dropna().unique().tolist())
marcas = sorted(productos_df["marca"].dropna().unique().tolist())
colores = sorted(productos_df["color"].dropna().unique().tolist())
voltajes = sorted(productos_df["voltaje"].dropna().unique().tolist())

precio_min = productos_df["precio_usd"].min()
precio_max = productos_df["precio_usd"].max()

stock_min = productos_df["stock"].min()
stock_max = productos_df["stock"].max()

potencia_min = productos_df["potencia_w"].min()
potencia_max = productos_df["potencia_w"].max()

garantia_min = productos_df["garantia_meses"].min()
garantia_max = productos_df["garantia_meses"].max()

resumen_tabular = f"""
Tabla principal: productos_df

Columnas disponibles:
- id_producto
- nombre
- categoria
- subcategoria
- marca
- precio_usd
- stock
- color
- potencia_w
- capacidad
- voltaje
- peso_kg
- garantia_meses
- descripcion

Valores posibles:
- categoria: {categorias}
- subcategoria: {subcategorias}
- marca: {marcas}
- color: {colores}
- voltaje: {voltajes}

Rangos numéricos:
- precio_usd: mínimo {precio_min}, máximo {precio_max}
- stock: mínimo {stock_min}, máximo {stock_max}
- potencia_w: mínimo {potencia_min}, máximo {potencia_max}
- garantia_meses: mínimo {garantia_min}, máximo {garantia_max}
"""

print(resumen_tabular)


Tabla principal: productos_df

Columnas disponibles:
- id_producto
- nombre
- categoria
- subcategoria
- marca
- precio_usd
- stock
- color
- potencia_w
- capacidad
- voltaje
- peso_kg
- garantia_meses
- descripcion

Valores posibles:
- categoria: ['Audio y Video', 'Climatización', 'Cocina', 'Lavado']
- subcategoria: ['Aires Acondicionados', 'Calefacción', 'Cocción', 'Lavado de Ropa', 'Lavado de Vajilla', 'Pequeños Electrodomésticos', 'Planchado', 'Preparación', 'Purificación', 'Refrigeración', 'Secado', 'Televisores', 'Ventilación']
- marca: ['AirFlow', 'ChefMaster', 'CleanMaster', 'ClimaTech', 'CookElite', 'EcoClima', 'FreshWash', 'HomeChef', 'KitchenPro', 'LaundryTech', 'PureAir', 'ScreenPro', 'SparkleHome', 'TechHome', 'ThermoControl', 'VisionPro', 'WashPro']
- color: ['Amarillo', 'Azul', 'Blanco', 'Dorado', 'Gris', 'Negro', 'Plateado', 'Rojo', 'Rosa', 'Verde']
- voltaje: ['110-220V', '12V', '220V']

Rangos numéricos:
- precio_usd: mínimo 28.22, máximo 2992.33
- stock: mínimo 1, m

### Interfaz de búsqueda tabular

In [20]:
def buscar_productos_tabular(filtros, top_k=10):
    # Se copia el DataFrame para no modificar la tabla original.
    df = productos_df.copy()

    # Filtro por texto contenido en el nombre del producto.
    if filtros.get("nombre_contiene"):
        texto = filtros["nombre_contiene"].lower()
        df = df[df["nombre"].str.lower().str.contains(texto, na=False)]

    # Filtro por categoría exacta.
    if filtros.get("categoria"):
        categoria = filtros["categoria"].lower()
        df = df[df["categoria"].str.lower() == categoria]

    # Filtro por subcategoría exacta.
    if filtros.get("subcategoria"):
        subcategoria = filtros["subcategoria"].lower()
        df = df[df["subcategoria"].str.lower() == subcategoria]

    # Filtro por marca exacta.
    if filtros.get("marca"):
        marca = filtros["marca"].lower()
        df = df[df["marca"].str.lower() == marca]

    # Filtro por precio máximo.
    if filtros.get("precio_max") is not None:
        df = df[df["precio_usd"] <= filtros["precio_max"]]

    # Filtro por precio mínimo.
    if filtros.get("precio_min") is not None:
        df = df[df["precio_usd"] >= filtros["precio_min"]]

    # Filtro por stock mínimo.
    if filtros.get("stock_min") is not None:
        df = df[df["stock"] >= filtros["stock_min"]]

    # Filtro por voltaje exacto.
    if filtros.get("voltaje"):
        voltaje = filtros["voltaje"].lower()
        df = df[df["voltaje"].str.lower() == voltaje]

    # Ordenamiento opcional.
    ordenar_por = filtros.get("ordenar_por")
    ascendente = filtros.get("ascendente", True)

    if ordenar_por in df.columns:
        df = df.sort_values(by=ordenar_por, ascending=ascendente)

    # Columnas que se devuelven como resultado.
    columnas_salida = [
        "id_producto", "nombre", "categoria", "subcategoria",
        "marca", "precio_usd", "stock", "voltaje", "garantia_meses"
    ]

    resultado = df[columnas_salida].head(top_k)

    if resultado.empty:
        print("No se encontraron productos con esos filtros.")

    return resultado

### Generación de filtros tabulares con LLM local

In [21]:
def extraer_json_desde_texto(texto):
    # Extrae el primer bloque con forma de JSON desde la respuesta del modelo.

    patron = re.search(r"\{.*\}", texto, re.DOTALL)

    if patron is None:
        return {}

    try:
        return json.loads(patron.group(0))
    except Exception:
        return {}

In [22]:
def limpiar_filtros_tabulares(filtros):
    # Deja solamente los filtros que acepta buscar_productos_tabular.
    # También convierte algunos valores numéricos para evitar errores.

    filtros_validos = {
        "nombre_contiene",
        "categoria",
        "subcategoria",
        "marca",
        "precio_max",
        "precio_min",
        "stock_min",
        "voltaje",
        "ordenar_por",
        "ascendente"
    }

    filtros_limpios = {}

    for clave, valor in filtros.items():
        # Si la clave no existe en nuestra función tabular, se ignora.
        if clave not in filtros_validos:
            continue

        # Si viene vacío, no se usa.
        if valor is None or valor == "":
            continue

        # Estos campos tienen que ser numéricos.
        if clave in ["precio_max", "precio_min", "stock_min"]:
            try:
                filtros_limpios[clave] = float(valor)
            except Exception:
                continue

        # ascendente tiene que ser booleano.
        elif clave == "ascendente":
            filtros_limpios[clave] = bool(valor)

        # ordenar_por solo se acepta si es una columna real del DataFrame.
        elif clave == "ordenar_por":
            if valor in productos_df.columns:
                filtros_limpios[clave] = valor

        else:
            valor_texto = str(valor).strip()

            # Ajuste simple para plural común.
            # Ejemplo: "licuadoras" -> "licuadora".
            if clave == "nombre_contiene" and valor_texto.lower().endswith("s"):
                valor_texto = valor_texto[:-1]

            filtros_limpios[clave] = valor_texto

    return filtros_limpios

In [23]:
def generar_filtros_tabulares_llm(consulta_usuario):
    # Convierte una consulta en lenguaje natural a filtros para buscar_productos_tabular.

    system_prompt = f"""
Sos un asistente que convierte consultas de usuarios en filtros JSON para buscar productos.

Tenés que devolver solamente un JSON válido, sin explicación y sin texto adicional.

Claves permitidas:
- nombre_contiene
- categoria
- subcategoria
- marca
- precio_max
- precio_min
- stock_min
- voltaje
- ordenar_por
- ascendente

Categorías posibles:
{categorias}

Subcategorías posibles:
{subcategorias}

Marcas posibles:
{marcas}

Voltajes posibles:
{voltajes}

Columnas posibles para ordenar:
{productos_df.columns.tolist()}

Ejemplo 1:
Consulta: ¿Cuáles son las licuadoras de menos de 400 dólares?
Respuesta:
{{"nombre_contiene": "licuadora", "precio_max": 400, "ordenar_por": "precio_usd", "ascendente": true}}

Ejemplo 2:
Consulta: Quiero productos TechHome con stock disponible
Respuesta:
{{"marca": "TechHome", "stock_min": 1}}

Ejemplo 3:
Consulta: Mostrame productos de cocina ordenados por precio
Respuesta:
{{"categoria": "Cocina", "ordenar_por": "precio_usd", "ascendente": true}}
"""

    user_prompt = f"""
Consulta del usuario:
{consulta_usuario}

Devolvé solamente el JSON.
"""

    respuesta = consultar_llm_local(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        max_new_tokens=180
    )

    filtros = extraer_json_desde_texto(respuesta)
    filtros = limpiar_filtros_tabulares(filtros)

    return filtros

In [24]:
def buscar_productos_natural(consulta_usuario, top_k=10):
    # Primero genera filtros con el LLM local.
    # Después usa esos filtros en la búsqueda tabular.

    filtros = generar_filtros_tabulares_llm(consulta_usuario)

    resultado = buscar_productos_tabular(
        filtros=filtros,
        top_k=top_k
    )

    return filtros, resultado

##  Base de datos de grafos con GrafitoDB

###  Preparación de relaciones desde DataFrame

In [25]:
# Se prepara un DataFrame de relaciones para la base de grafos.
# Cada fila representa una relación entre dos nodos.

relaciones = []

for _, row in productos_df.iterrows():
    id_producto = row["id_producto"]
    nombre_producto = row["nombre"]
    categoria = row["categoria"]
    subcategoria = row["subcategoria"]
    marca = row["marca"]

    # Producto -> Categoría
    relaciones.append({
        "origen_tipo": "Producto",
        "origen_id": id_producto,
        "origen_nombre": nombre_producto,
        "relacion": "PERTENECE_A",
        "destino_tipo": "Categoria",
        "destino_id": categoria,
        "destino_nombre": categoria
    })

    # Producto -> Subcategoría
    relaciones.append({
        "origen_tipo": "Producto",
        "origen_id": id_producto,
        "origen_nombre": nombre_producto,
        "relacion": "TIENE_SUBCATEGORIA",
        "destino_tipo": "Subcategoria",
        "destino_id": subcategoria,
        "destino_nombre": subcategoria
    })

    # Producto -> Marca
    relaciones.append({
        "origen_tipo": "Producto",
        "origen_id": id_producto,
        "origen_nombre": nombre_producto,
        "relacion": "TIENE_MARCA",
        "destino_tipo": "Marca",
        "destino_id": marca,
        "destino_nombre": marca
    })

relaciones_grafo_df = pd.DataFrame(relaciones)

print("Relaciones preparadas:", relaciones_grafo_df.shape)
display(relaciones_grafo_df.head(10))

Relaciones preparadas: (900, 7)


,origen_tipo,origen_id,origen_nombre,relacion,destino_tipo,destino_id,destino_nombre
0,Producto,P0001,Licuadora,PERTENECE_A,Categoria,Cocina,Cocina
1,Producto,P0001,Licuadora,TIENE_SUBCATEGORIA,Subcategoria,Preparación,Preparación
2,Producto,P0001,Licuadora,TIENE_MARCA,Marca,TechHome,TechHome
3,Producto,P0002,Licuadora,PERTENECE_A,Categoria,Cocina,Cocina
4,Producto,P0002,Licuadora,TIENE_SUBCATEGORIA,Subcategoria,Preparación,Preparación
5,Producto,P0002,Licuadora,TIENE_MARCA,Marca,TechHome,TechHome
6,Producto,P0003,Plus Licuadora Pro,PERTENECE_A,Categoria,Cocina,Cocina
7,Producto,P0003,Plus Licuadora Pro,TIENE_SUBCATEGORIA,Subcategoria,Preparación,Preparación
8,Producto,P0003,Plus Licuadora Pro,TIENE_MARCA,Marca,TechHome,TechHome
9,Producto,P0004,Compacto Licuadora,PERTENECE_A,Categoria,Cocina,Cocina


### Creación de la base GrafitoDB

In [26]:
# Se crea una base de grafos en memoria.
# La base queda disponible mientras dure la sesión de Colab.

db_grafo = GrafitoDatabase(':memory:', cypher_max_hops=6)

print("Base GrafitoDB creada.")

Base GrafitoDB creada.


### Carga de nodos y relaciones en GrafitoDB

In [27]:
# Diccionario auxiliar para no crear nodos duplicados.
# La clave será una combinación de tipo de nodo e id.

nodos_grafo = {}

def obtener_o_crear_nodo(tipo, nodo_id, nombre):
    # Se arma una clave única para cada nodo.
    clave = (tipo, str(nodo_id))

    # Si el nodo ya fue creado, se reutiliza.
    if clave in nodos_grafo:
        return nodos_grafo[clave]

    # Si no existe, se crea en GrafitoDB.
    nodo = db_grafo.create_node(
        labels=[tipo],
        properties={
            "id": str(nodo_id),
            "nombre": str(nombre)
        }
    )

    nodos_grafo[clave] = nodo
    return nodo


# Se cargan en GrafitoDB las relaciones preparadas en relaciones_grafo_df.
for _, row in relaciones_grafo_df.iterrows():
    origen = obtener_o_crear_nodo(
        row["origen_tipo"],
        row["origen_id"],
        row["origen_nombre"]
    )

    destino = obtener_o_crear_nodo(
        row["destino_tipo"],
        row["destino_id"],
        row["destino_nombre"]
    )

    db_grafo.create_relationship(
        origen.id,
        destino.id,
        row["relacion"]
    )

print("Nodos creados:", len(nodos_grafo))
print("Relaciones cargadas:", len(relaciones_grafo_df))

Nodos creados: 334
Relaciones cargadas: 900


### Interfaz de consulta para GrafitoDB

In [28]:
def buscar_en_grafo(query_cypher, limite=10):
    # Ejecuta una consulta Cypher sobre GrafitoDB.
    # Devuelve como máximo la cantidad de resultados indicada en limite.

    resultado = db_grafo.execute(query_cypher)

    return resultado[:limite]

### Generación de consultas Cypher con LLM local

In [29]:
def extraer_cypher_desde_texto(texto):
    # Limpia la respuesta del modelo y extrae una consulta Cypher.

    texto = texto.replace("```cypher", "")
    texto = texto.replace("```", "")
    texto = texto.strip()

    inicio = texto.upper().find("MATCH")

    if inicio == -1:
        return None

    query = texto[inicio:].strip()

    return query


def validar_cypher_lectura(query_cypher):
    # Valida que la consulta sea solo de lectura.
    # Esto evita ejecutar instrucciones que modifiquen el grafo.

    if query_cypher is None:
        return False

    query_upper = query_cypher.upper()

    palabras_prohibidas = [
        "CREATE",
        "MERGE",
        "DELETE",
        "SET ",
        "REMOVE",
        "DROP",
        "CALL",
        "LOAD"
    ]

    if not query_upper.startswith("MATCH"):
        return False

    if "RETURN" not in query_upper:
        return False

    for palabra in palabras_prohibidas:
        if palabra in query_upper:
            return False

    return True

In [30]:
def generar_cypher_llm(consulta_usuario):
    # Convierte una pregunta en lenguaje natural a una consulta Cypher.
    # Usa solamente el esquema del grafo creado en GrafitoDB.

    system_prompt = f"""
Sos un asistente que convierte preguntas de usuarios en consultas Cypher para GrafitoDB.

Tenés que devolver solamente una consulta Cypher, sin explicación y sin texto adicional.

El grafo tiene estos nodos:
(:Producto {{id, nombre}})
(:Categoria {{id, nombre}})
(:Subcategoria {{id, nombre}})
(:Marca {{id, nombre}})

Relaciones disponibles:
(:Producto)-[:PERTENECE_A]->(:Categoria)
(:Producto)-[:TIENE_SUBCATEGORIA]->(:Subcategoria)
(:Producto)-[:TIENE_MARCA]->(:Marca)

Categorías posibles:
{categorias}

Subcategorías posibles:
{subcategorias}

Marcas posibles:
{marcas}

Reglas:
- Usá solamente MATCH y RETURN.
- No uses CREATE, MERGE, DELETE, SET ni ninguna consulta que modifique datos.
- Si preguntan por categoría, usá PERTENECE_A.
- Si preguntan por subcategoría, usá TIENE_SUBCATEGORIA.
- Si preguntan por marca, usá TIENE_MARCA.

Ejemplo 1:
Pregunta: ¿Qué productos están relacionados con la categoría Cocina?
Respuesta:
MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {{nombre: 'Cocina'}})
RETURN p.nombre, c.nombre

Ejemplo 2:
Pregunta: ¿Qué productos son de la marca TechHome?
Respuesta:
MATCH (p:Producto)-[:TIENE_MARCA]->(m:Marca {{nombre: 'TechHome'}})
RETURN p.nombre, m.nombre

Ejemplo 3:
Pregunta: ¿Qué productos pertenecen a la subcategoría Preparación?
Respuesta:
MATCH (p:Producto)-[:TIENE_SUBCATEGORIA]->(s:Subcategoria {{nombre: 'Preparación'}})
RETURN p.nombre, s.nombre
"""

    user_prompt = f"""
Pregunta del usuario:
{consulta_usuario}

Devolvé solamente la consulta Cypher.
"""

    respuesta = consultar_llm_local(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        max_new_tokens=180
    )

    query_cypher = extraer_cypher_desde_texto(respuesta)

    if not validar_cypher_lectura(query_cypher):
        return None

    return query_cypher

In [31]:
def buscar_en_grafo_natural(consulta_usuario, limite=10):
    # Primero genera una consulta Cypher con el LLM local.
    # Después ejecuta esa consulta sobre GrafitoDB.

    query_cypher = generar_cypher_llm(consulta_usuario)

    if query_cypher is None:
        return {
            "query_cypher": None,
            "resultados": []
        }

    resultados = buscar_en_grafo(
        query_cypher=query_cypher,
        limite=limite
    )

    return {
        "query_cypher": query_cypher,
        "resultados": resultados
    }

## Clasificador de intención avanzado

###  Dataset sintético de preguntas

In [32]:
# Dataset sintético de preguntas para entrenar un clasificador de intención.
# Cada ejemplo tiene una pregunta y la fuente que debería consultar el sistema.

datos_intencion = [
    # Consultas vectoriales: manuales, FAQs, reseñas, tickets o problemas descriptivos.
    ("¿Cómo uso mi licuadora para hacer smoothies?", "vectorial"),
    ("¿Qué opinan los usuarios de esta cafetera?", "vectorial"),
    ("Quiero una licuadora con buenas reseñas", "vectorial"),
    ("¿Qué problemas reportaron los clientes sobre heladeras?", "vectorial"),
    ("¿Cómo se limpia una aspiradora según el manual?", "vectorial"),
    ("¿Qué dice el manual sobre la instalación del lavarropas?", "vectorial"),
    ("¿Hay reseñas negativas sobre cafeteras?", "vectorial"),
    ("¿Qué comentarios hicieron los usuarios sobre microondas?", "vectorial"),
    ("¿Cómo configuro el modo eco de un aire acondicionado?", "vectorial"),
    ("¿Qué fallas aparecen en los tickets de soporte?", "vectorial"),
    ("Necesito instrucciones para usar una procesadora", "vectorial"),
    ("¿Qué recomendaciones hay para cuidar una licuadora?", "vectorial"),

    # Consultas tabulares: filtros sobre precio, stock, marca, garantía, voltaje o productos.
    ("¿Cuáles son las licuadoras de menos de 400 dólares con stock?", "tabular"),
    ("Mostrame productos TechHome con stock disponible", "tabular"),
    ("¿Qué productos cuestan menos de 200 dólares?", "tabular"),
    ("Quiero cafeteras ordenadas por precio", "tabular"),
    ("¿Cuáles productos tienen garantía de más de 24 meses?", "tabular"),
    ("Mostrame productos de cocina con precio bajo", "tabular"),
    ("¿Hay aspiradoras con stock mayor a 10?", "tabular"),
    ("Buscá productos de 220V", "tabular"),
    ("Quiero una licuadora barata disponible", "tabular"),
    ("¿Qué productos hay de la marca ChefMaster?", "tabular"),
    ("Mostrame productos con precio entre 100 y 500 dólares", "tabular"),
    ("¿Qué heladeras tienen stock?", "tabular"),

    # Consultas de grafo: relaciones entre producto, categoría, subcategoría y marca.
    ("¿Qué productos están relacionados con la categoría Cocina?", "grafo"),
    ("¿Qué productos son de la marca TechHome?", "grafo"),
    ("¿Qué productos pertenecen a la subcategoría Preparación?", "grafo"),
    ("Mostrame productos conectados con la marca ChefMaster", "grafo"),
    ("¿Qué productos pertenecen a la categoría Limpieza?", "grafo"),
    ("¿Qué productos están vinculados con la subcategoría Refrigeración?", "grafo"),
    ("Listame productos relacionados con una marca", "grafo"),
    ("¿Qué productos están conectados a Cocina?", "grafo"),
    ("¿Qué productos se relacionan con TechHome?", "grafo"),
    ("¿Qué productos pertenecen a cada categoría?", "grafo"),
    ("Mostrame relaciones entre productos y marcas", "grafo"),
    ("Quiero consultar conexiones entre productos y subcategorías", "grafo"),
]

intenciones_df = pd.DataFrame(datos_intencion, columns=["consulta", "intencion"])

display(intenciones_df.head())
print(intenciones_df["intencion"].value_counts())

,consulta,intencion
0,¿Cómo uso mi licuadora para hacer smoothies?,vectorial
1,¿Qué opinan los usuarios de esta cafetera?,vectorial
2,Quiero una licuadora con buenas reseñas,vectorial
3,¿Qué problemas reportaron los clientes sobre h...,vectorial
4,¿Cómo se limpia una aspiradora según el manual?,vectorial


intencion
vectorial    12
tabular      12
grafo        12
Name: count, dtype: int64


### Entrenamiento del clasificador supervisado

In [33]:
# Se separan las consultas y las etiquetas.
X = intenciones_df["consulta"]
y = intenciones_df["intencion"]

# Se divide el dataset en entrenamiento y prueba.
# stratify=y mantiene proporciones parecidas de cada intención en ambos conjuntos.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Ejemplos de entrenamiento:", len(X_train))
print("Ejemplos de prueba:", len(X_test))
print("\nDistribución en entrenamiento:")
print(y_train.value_counts())
print("\nDistribución en prueba:")
print(y_test.value_counts())

Ejemplos de entrenamiento: 25
Ejemplos de prueba: 11

Distribución en entrenamiento:
intencion
grafo        9
vectorial    8
tabular      8
Name: count, dtype: int64

Distribución en prueba:
intencion
vectorial    4
tabular      4
grafo        3
Name: count, dtype: int64


In [34]:
# Pipeline de clasificación:
# primero TF-IDF convierte texto en vectores numéricos.
# segund LogisticRegression aprende a clasificar la intención.

clasificador_intencion = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

clasificador_intencion.fit(X_train, y_train)

print("Clasificador entrenado.")

Clasificador entrenado.


In [35]:
y_pred = clasificador_intencion.predict(X_test)

print(classification_report(y_test, y_pred))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred, labels=["vectorial", "tabular", "grafo"]))

              precision    recall  f1-score   support

       grafo       1.00      1.00      1.00         3
     tabular       1.00      1.00      1.00         4
   vectorial       1.00      1.00      1.00         4

    accuracy                           1.00        11
   macro avg       1.00      1.00      1.00        11
weighted avg       1.00      1.00      1.00        11

Matriz de confusión:
[[4 0 0]
 [0 4 0]
 [0 0 3]]


In [36]:
def clasificar_intencion_supervisado(consulta_usuario):
    # Usa el clasificador entrenado para predecir la intención de una consulta nueva.

    intencion = clasificador_intencion.predict([consulta_usuario])[0]

    probabilidades = clasificador_intencion.predict_proba([consulta_usuario])[0]
    clases = clasificador_intencion.classes_

    scores = {
        clase: round(float(prob), 4)
        for clase, prob in zip(clases, probabilidades)
    }

    return intencion, scores

### Clasificador de intención con LLM local y few-shot prompting

In [37]:
def clasificar_intencion_llm(consulta_usuario):
    # Clasifica la intención usando el modelo local con ejemplos en el prompt.

    system_prompt = """
Sos un clasificador de intención para un asistente de una empresa de electrodomésticos.

Tenés que clasificar cada consulta en una sola de estas etiquetas:

vectorial: cuando la consulta se responde con manuales, FAQs, reseñas, opiniones, tickets o textos descriptivos.
tabular: cuando la consulta requiere filtros sobre productos, precios, stock, garantía, voltaje o datos estructurados.
grafo: cuando la consulta pregunta por relaciones entre productos, categorías, subcategorías o marcas.

Devolvé solamente una etiqueta: vectorial, tabular o grafo.
No expliques nada.

Ejemplos:

Consulta: ¿Cómo uso mi licuadora para hacer smoothies?
Intención: vectorial

Consulta: ¿Qué opinan los usuarios de esta cafetera?
Intención: vectorial

Consulta: Quiero una licuadora con buenas reseñas
Intención: vectorial

Consulta: ¿Cuáles son las licuadoras de menos de 400 dólares?
Intención: tabular

Consulta: Mostrame productos TechHome con stock disponible
Intención: tabular

Consulta: ¿Qué productos cuestan menos de 200 dólares?
Intención: tabular

Consulta: ¿Qué productos están relacionados con la categoría Cocina?
Intención: grafo

Consulta: ¿Qué productos son de la marca TechHome?
Intención: grafo

Consulta: ¿Qué productos pertenecen a la subcategoría Preparación?
Intención: grafo
"""

    user_prompt = f"""
Consulta: {consulta_usuario}
Intención:
"""

    respuesta = consultar_llm_local(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        max_new_tokens=20
    )

    respuesta = respuesta.lower().strip()

    # Se normaliza la respuesta por si el modelo agrega algún texto extra. Sino me imprime None
    if "vectorial" in respuesta:
        return "vectorial"
    elif "tabular" in respuesta:
        return "tabular"
    elif "grafo" in respuesta:
        return "grafo"
    else:
        return "desconocida"

###  Comparación de clasificadores de intención

In [38]:
# Se evalúa el clasificador con LLM local sobre el mismo conjunto de prueba
# usado para el clasificador supervisado.

y_pred_llm = []

for consulta in X_test:
    intencion = clasificar_intencion_llm(consulta)
    y_pred_llm.append(intencion)

print("Evaluación del clasificador LLM few-shot:")
print(classification_report(y_test, y_pred_llm))

print("Matriz de confusión LLM:")
print(confusion_matrix(y_test, y_pred_llm, labels=["vectorial", "tabular", "grafo"]))

Evaluación del clasificador LLM few-shot:
              precision    recall  f1-score   support

       grafo       1.00      1.00      1.00         3
     tabular       1.00      1.00      1.00         4
   vectorial       1.00      1.00      1.00         4

    accuracy                           1.00        11
   macro avg       1.00      1.00      1.00        11
weighted avg       1.00      1.00      1.00        11

Matriz de confusión LLM:
[[4 0 0]
 [0 4 0]
 [0 0 3]]


In [39]:
comparacion_intenciones = pd.DataFrame({
    "consulta": X_test.values,
    "intencion_real": y_test.values,
    "pred_supervisado": y_pred,
    "pred_llm": y_pred_llm
})

display(comparacion_intenciones)

,consulta,intencion_real,pred_supervisado,pred_llm
0,¿Qué recomendaciones hay para cuidar una licua...,vectorial,vectorial,vectorial
1,¿Qué dice el manual sobre la instalación del l...,vectorial,vectorial,vectorial
2,¿Qué productos cuestan menos de 200 dólares?,tabular,tabular,tabular
3,¿Qué fallas aparecen en los tickets de soporte?,vectorial,vectorial,vectorial
4,¿Qué productos están conectados a Cocina?,grafo,grafo,grafo
5,Mostrame productos de cocina con precio bajo,tabular,tabular,tabular
6,¿Qué productos pertenecen a la categoría Limpi...,grafo,grafo,grafo
7,¿Qué heladeras tienen stock?,tabular,tabular,tabular
8,Mostrame relaciones entre productos y marcas,grafo,grafo,grafo
9,¿Cómo configuro el modo eco de un aire acondic...,vectorial,vectorial,vectorial


Ambos clasificadores obtuvieron buen rendimiento sobre el conjunto de prueba sintético. El clasificador supervisado es más rápido y estable, ya que no necesita generar texto en cada consulta. Sin embargo, el clasificador con LLM local mediante few-shot prompting resulta más flexible para interpretar consultas redactadas de distintas formas.

Para el pipeline principal se utiliza el clasificador con LLM local, ya que el sistema trabaja con consultas en lenguaje natural y se prioriza la flexibilidad en la interpretación de la intención. El clasificador supervisado queda como comparación entrenada con preguntas sintéticas.

##  Pipeline de recuperación

###  Índice BM25 para búsqueda por palabras clave

In [40]:
# Se crea un índice BM25 sobre los mismos fragmentos que ya se cargaron en ChromaDB.
# BM25 sirve para recuperar textos por coincidencia de palabras clave.

scoring_bm25 = ScoringFactory.create({
    "method": "bm25",
    "terms": True
})

# txtai espera elementos con formato:
# (id_numérico, texto, tags)
# Los tags no se usan en este caso.
scoring_bm25.index(
    (i, texto, None) for i, texto in enumerate(documents)
)

print("Índice BM25 creado sobre", len(documents), "fragmentos.")

Índice BM25 creado sobre 10575 fragmentos.


###  Función de búsqueda BM25

In [41]:
def buscar_bm25(consulta, k=5):
    # Busca fragmentos usando BM25.
    # Devuelve una lista de resultados con texto, metadata y score.

    resultados = scoring_bm25.search(consulta, k)

    salida = []

    for indice, score in resultados:
        salida.append({
            "id": ids[indice],
            "texto": documents[indice],
            "metadata": metadatas[indice],
            "score_bm25": score
        })

    return salida

###  Función de búsqueda semántica normalizada

In [42]:
def buscar_semantico_normalizado(consulta, k=5, filtros=None):
    # Usa la búsqueda semántica de ChromaDB.
    # Devuelve los resultados en una estructura similar a BM25
    # para poder combinarlos después.

    resultados = buscar_en_chroma(
        consulta=consulta,
        k=k,
        filtros=filtros
    )

    salida = []

    cantidad = len(resultados["documents"][0])

    for i in range(cantidad):
        salida.append({
            "id": resultados["ids"][0][i],
            "texto": resultados["documents"][0][i],
            "metadata": resultados["metadatas"][0][i],
            "distancia_semantica": resultados["distances"][0][i]
        })

    return salida

###  Búsqueda híbrida semántica + BM25

In [43]:
def busqueda_hibrida(consulta, k_semantico=8, k_bm25=8, filtros=None):
    # Recupera candidatos desde dos métodos:
    # ChromaDB: búsqueda semántica.
    # BM25: búsqueda por palabras clave.

    resultados_semanticos = buscar_semantico_normalizado(
        consulta=consulta,
        k=k_semantico,
        filtros=filtros
    )

    resultados_bm25 = buscar_bm25(
        consulta=consulta,
        k=k_bm25
    )

    candidatos = {}

    # Se agregan los resultados semánticos.
    for r in resultados_semanticos:
        candidatos[r["id"]] = {
            "id": r["id"],
            "texto": r["texto"],
            "metadata": r["metadata"],
            "distancia_semantica": r["distancia_semantica"],
            "score_bm25": None
        }

    # Se agregan los resultados BM25.
    # Si el mismo fragmento ya estaba, se completa el score BM25.
    for r in resultados_bm25:
        if r["id"] not in candidatos:
            candidatos[r["id"]] = {
                "id": r["id"],
                "texto": r["texto"],
                "metadata": r["metadata"],
                "distancia_semantica": None,
                "score_bm25": r["score_bm25"]
            }
        else:
            candidatos[r["id"]]["score_bm25"] = r["score_bm25"]

    return list(candidatos.values())

### Re-rank de candidatos recuperados

In [44]:
def rerank_candidatos(consulta, candidatos, top_k=5):
    # Reordena los candidatos usando similitud semántica con embeddings.
    # Primero se recuperan candidatos con búsqueda híbrida y después se ordenan otra vez.

    if len(candidatos) == 0:
        return []

    textos_candidatos = [c["texto"] for c in candidatos]

    consulta_embedding = embedding_model.encode(
        "query: " + consulta,
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    candidatos_embeddings = embedding_model.encode(
        ["passage: " + texto for texto in textos_candidatos],
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    similitudes = util.cos_sim(
        consulta_embedding,
        candidatos_embeddings
    )[0]

    resultados = []

    for i, candidato in enumerate(candidatos):
        nuevo = candidato.copy()
        nuevo["score_rerank"] = round(float(similitudes[i]), 4)
        resultados.append(nuevo)

    resultados = sorted(
        resultados,
        key=lambda x: x["score_rerank"],
        reverse=True
    )

    return resultados[:top_k]

###  Función final de recuperación textual

In [45]:
def recuperar_textos(consulta, top_k=5, filtros=None):
    # Pipeline completo para recuperar textos:
    # 1) búsqueda semántica con ChromaDB
    # 2) búsqueda por palabras clave con BM25
    # 3) unión de candidatos
    # 4) re-rank final con embeddings

    candidatos = busqueda_hibrida(
        consulta=consulta,
        k_semantico=10,
        k_bm25=10,
        filtros=filtros
    )

    resultados = rerank_candidatos(
        consulta=consulta,
        candidatos=candidatos,
        top_k=top_k
    )

    return resultados

In [46]:
def limpiar_datos_personales(texto):
    # Oculta teléfonos y usuarios para no mostrar datos personales en las salidas finales.

    texto = re.sub(r"Teléfono:\s*.*", "Teléfono: [oculto]", texto)
    texto = re.sub(r"Usuario:\s*.*", "Usuario: [oculto]", texto)

    return texto

###  Pipeline general de recuperación

In [47]:
def pipeline_recuperacion(consulta_usuario, top_k=5):
    # Clasifica la intención de la consulta.
    # Según la intención detectada, consulta la fuente correspondiente.

    intencion = clasificar_intencion_llm(consulta_usuario)

    salida = {
        "consulta": consulta_usuario,
        "intencion": intencion,
        "fuente": None,
        "consulta_generada": None,
        "resultados": None
    }

    if intencion == "vectorial":
        resultados = recuperar_textos(
            consulta=consulta_usuario,
            top_k=top_k
        )

        salida["fuente"] = "ChromaDB + BM25 + ReRank"
        salida["resultados"] = resultados

    elif intencion == "tabular":
        filtros, resultado_tabular = buscar_productos_natural(
            consulta_usuario=consulta_usuario,
            top_k=top_k
        )

        salida["fuente"] = "Pandas"
        salida["consulta_generada"] = filtros
        salida["resultados"] = resultado_tabular

    elif intencion == "grafo":
        resultado_grafo = buscar_en_grafo_natural(
            consulta_usuario=consulta_usuario,
            limite=top_k
        )

        salida["fuente"] = "GrafitoDB"
        salida["consulta_generada"] = resultado_grafo["query_cypher"]
        salida["resultados"] = resultado_grafo["resultados"]

    else:
        salida["fuente"] = "desconocida"
        salida["resultados"] = []

    return salida

##  Checkpoint de recuperación

### Función para mostrar resultados recuperados

In [48]:
def mostrar_resultado_pipeline(resultado):
    # Muestra de forma ordenada la salida del pipeline de recuperación.
    # Todavía no genera respuesta final con LLM.

    print("Consulta:", resultado["consulta"])
    print("Intención detectada:", resultado["intencion"])
    print("Fuente seleccionada:", resultado["fuente"])
    print("Consulta generada:", resultado["consulta_generada"])
    print("\nResultados recuperados:")

    if resultado["intencion"] == "vectorial":
        for i, r in enumerate(resultado["resultados"], start=1):
            print(f"\nResultado {i}")
            print("ID:", r["id"])
            print("Fuente:", r["metadata"].get("fuente"))
            print("Tipo:", r["metadata"].get("tipo"))
            print("Score rerank:", r.get("score_rerank"))
            print("Distancia semántica:", r.get("distancia_semantica"))
            print("Score BM25:", r.get("score_bm25"))
            print("Texto:", limpiar_datos_personales(r["texto"])[:600])

    elif resultado["intencion"] == "tabular":
        display(resultado["resultados"])

    elif resultado["intencion"] == "grafo":
        for i, fila in enumerate(resultado["resultados"], start=1):
            print(f"Resultado {i}:", fila)

    else:
        print("No se encontraron resultados o la intención no pudo ser clasificada.")

In [49]:
prompts_checkpoint = [
    "¿Cómo uso mi licuadora para hacer smoothies?",
    "¿Cuáles son las licuadoras de menos de 200 dólares?",
    "¿Qué opinan los usuarios de esta cafetera?",
    "Quiero una licuadora con buenas reseñas",
    "¿Qué productos están relacionados con la categoría Cocina?"
]

for i, prompt in enumerate(prompts_checkpoint, start=1):
    print("=" * 100)
    print(f"PROMPT {i}")
    print("=" * 100)

    resultado_checkpoint = pipeline_recuperacion(
        prompt,
        top_k=3
    )

    mostrar_resultado_pipeline(resultado_checkpoint)
    print("\n")

PROMPT 1
Consulta: ¿Cómo uso mi licuadora para hacer smoothies?
Intención detectada: vectorial
Fuente seleccionada: ChromaDB + BM25 + ReRank
Consulta generada: None

Resultados recuperados:

Resultado 1
ID: faq_14_chunk_0
Fuente: faqs
Tipo: pregunta_frecuente
Score rerank: 0.8615
Distancia semántica: 0.2770891785621643
Score BM25: None
Texto: FAQ00015 P0002 Licuadora Uso ¿Puedo usarlo todos los días? El Licuadora de TechHome está diseñado para uso doméstico. Revise el manual del producto (código P0002) para más detalles. Ante cualquier duda, contacte a nuestro servicio de atención al cliente. 2025-02-27 994 77

Resultado 2
ID: faq_6_chunk_0
Fuente: faqs
Tipo: pregunta_frecuente
Score rerank: 0.8596
Distancia semántica: 0.2807939946651459
Score BM25: 11.59107494354248
Texto: FAQ00007 P0001 Licuadora Uso ¿Cómo se usa correctamente este producto? El Licuadora de TechHome está diseñado para uso doméstico. Revise el manual del producto (código P0001) para más detalles. Ante cualquier duda, 

,id_producto,nombre,categoria,subcategoria,marca,precio_usd,stock,voltaje,garantia_meses




PROMPT 3
Consulta: ¿Qué opinan los usuarios de esta cafetera?
Intención detectada: vectorial
Fuente seleccionada: ChromaDB + BM25 + ReRank
Consulta generada: None

Resultados recuperados:

Resultado 1
ID: resena_resena_R04450_chunk_0
Fuente: resenas_usuarios
Tipo: resena
Score rerank: 0.8662
Distancia semántica: 0.26765725016593933
Score BM25: None
Texto: Fecha: 2025-07-30
Usuario: [oculto]
Teléfono: [oculto]
Producto: Cafetera (P0122)
Puntaje: 3/5
Provincia: Santa Fe
Les cuento... Tiene sus pros y contras con Cafetera. Tiene cosas buenas y cosas malas. Por el precio está bien, pero no esperen maravillas. Espero les sirva!

Resultado 2
ID: resena_resena_R00776_chunk_0
Fuente: resenas_usuarios
Tipo: resena
Score rerank: 0.8632
Distancia semántica: 0.27358895540237427
Score BM25: None
Texto: Fecha: 2024-06-09
Usuario: [oculto]
Teléfono: [oculto]
Producto: Cafetera (P0122)
Puntaje: 4/5
Provincia: Río Negro
Les cuento... Increíble relación calidad-precio con Cafetera. Lo compré hace dos 

##  Generación final de respuestas

### Preparación del contexto recuperado

In [50]:
def formatear_contexto_para_respuesta(resultado_pipeline):
    # Convierte los resultados recuperados en texto para pasárselos al LLM.
    # La idea es que el modelo responda usando solo este contexto.

    intencion = resultado_pipeline["intencion"]
    contexto = ""

    if intencion == "vectorial":
        partes = []

        for i, r in enumerate(resultado_pipeline["resultados"], start=1):
            texto_limpio = limpiar_datos_personales(r["texto"])

            partes.append(
                f"""
Resultado {i}
Fuente: {r["metadata"].get("fuente")}
Tipo: {r["metadata"].get("tipo")}
Score rerank: {r.get("score_rerank")}
Texto:
{texto_limpio}
"""
            )

        contexto = "\n".join(partes)

    elif intencion == "tabular":
        df = resultado_pipeline["resultados"]

        if df is None or df.empty:
            contexto = "No se encontraron registros tabulares para la consulta."
        else:
            filas = []

            for _, row in df.iterrows():
                filas.append(
                    f"""
Producto encontrado:
- id_producto: {row["id_producto"]}
- nombre: {row["nombre"]}
- categoria: {row["categoria"]}
- subcategoria: {row["subcategoria"]}
- marca: {row["marca"]}
- precio_usd: {row["precio_usd"]}
- stock: {row["stock"]}
- voltaje: {row["voltaje"]}
- garantia_meses: {row["garantia_meses"]}
"""
                )

            contexto = "\n".join(filas)

    elif intencion == "grafo":
        resultados = resultado_pipeline["resultados"]

        if len(resultados) == 0:
            contexto = "No se encontraron resultados en el grafo."
        else:
            contexto = "\n".join([str(fila) for fila in resultados])

    else:
        contexto = "No se pudo determinar una fuente de información."

    return contexto

In [51]:
resultado_prueba = pipeline_recuperacion(
    "¿Qué opinan los usuarios de esta cafetera?",
    top_k=3
)

contexto_prueba = formatear_contexto_para_respuesta(resultado_prueba)

print(contexto_prueba[:1500])


Resultado 1
Fuente: resenas_usuarios
Tipo: resena
Score rerank: 0.8662
Texto:
Fecha: 2025-07-30
Usuario: [oculto]
Teléfono: [oculto]
Producto: Cafetera (P0122)
Puntaje: 3/5
Provincia: Santa Fe
Les cuento... Tiene sus pros y contras con Cafetera. Tiene cosas buenas y cosas malas. Por el precio está bien, pero no esperen maravillas. Espero les sirva!


Resultado 2
Fuente: resenas_usuarios
Tipo: resena
Score rerank: 0.8632
Texto:
Fecha: 2024-06-09
Usuario: [oculto]
Teléfono: [oculto]
Producto: Cafetera (P0122)
Puntaje: 4/5
Provincia: Río Negro
Les cuento... Increíble relación calidad-precio con Cafetera. Lo compré hace dos meses y es muy duradero, además de elegante. Mi familia está muy contenta con la compra. Espero les sirva!


Resultado 3
Fuente: resenas_usuarios
Tipo: resena
Score rerank: 0.8624
Texto:
Fecha: 2025-10-03
Usuario: [oculto]
Teléfono: [oculto]
Producto: Cafetera (P0122)
Puntaje: 4/5
Provincia: Chaco
Les cuento... Excelente producto, superó mis expectativas con Cafetera. 

In [52]:
def generar_respuesta_tabular(resultado_pipeline):
    # Genera una respuesta tabular sin LLM para evitar que el modelo omita productos o invente datos.
    # Se usa únicamente lo que devolvió Pandas en el pipeline de recuperación.

    df = resultado_pipeline["resultados"]

    if df is None or df.empty:
        return "No se encontraron productos que cumplan con esos filtros."

    lineas = []
    lineas.append(f"Se encontraron {len(df)} productos que cumplen con la consulta:")

    for _, row in df.iterrows():
        lineas.append(
            f"- {row['nombre']} ({row['marca']}): "
            f"precio USD {row['precio_usd']}, "
            f"stock {row['stock']}, "
            f"voltaje {row['voltaje']}, "
            f"garantía {row['garantia_meses']} meses."
        )

    return "\n".join(lineas)

In [53]:
def generar_respuesta_grafo(resultado_pipeline):
    # Genera una respuesta para resultados de grafo sin LLM.
    # Se usa únicamente lo que devolvió GrafitoDB.

    resultados = resultado_pipeline["resultados"]

    if resultados is None or len(resultados) == 0:
        return "No se encontraron relaciones en el grafo para esa consulta."

    nombres_productos = []

    for fila in resultados:
        if "p.nombre" in fila:
            nombres_productos.append(fila["p.nombre"])

    productos_unicos = []
    for producto in nombres_productos:
        if producto not in productos_unicos:
            productos_unicos.append(producto)

    lineas = []
    lineas.append("Los productos únicos recuperados desde el grafo son:")

    for producto in productos_unicos:
        lineas.append(f"- {producto}")

    return "\n".join(lineas)

### Generación de respuesta con LLM local

In [54]:
def generar_respuesta_final(consulta_usuario, resultado_pipeline):
    # Genera una respuesta final usando la consulta original y el contexto recuperado.
    # La respuesta se adapta según la fuente consultada.

    contexto = formatear_contexto_para_respuesta(resultado_pipeline)
    intencion = resultado_pipeline["intencion"]

    if intencion == "tabular":
        return generar_respuesta_tabular(resultado_pipeline)

    elif intencion == "grafo":
        return generar_respuesta_grafo(resultado_pipeline)

    elif intencion == "vectorial":
        instrucciones_especificas = """
La consulta fue respondida con textos recuperados desde la base vectorial.

Si el contexto contiene reseñas:
- Resumí solamente las opiniones que aparecen en las reseñas recuperadas.
- Separá opiniones positivas, mixtas o negativas si corresponde.
- No generalices diciendo que todos los usuarios piensan algo.
- Usá frases como "según las reseñas recuperadas".
- No calcules promedios ni porcentajes salvo que aparezcan explícitamente calculados en el contexto.
- No conviertas una opinión individual en una tendencia general. Si una crítica aparece en una sola reseña, aclaralo como "una reseña menciona...".
- Redactá en lenguaje natural, evitando frases literales raras.

Si el contexto contiene FAQs, manuales o tickets:
- Respondé usando únicamente las instrucciones, aclaraciones o problemas mencionados.
- Si no aparece una respuesta exacta, decí que no se encontró información específica y mencioná lo más cercano.
"""

    else:
        instrucciones_especificas = """
No se pudo determinar una fuente clara de información.
Decí que no hay información suficiente para responder.
"""

    system_prompt = f"""
Sos un asistente virtual de una empresa de electrodomésticos.

Respondé en español, de forma clara y breve.

Reglas obligatorias:
- Usá solamente la información que aparece explícitamente en el contexto recuperado.
- No inventes datos.
- No agregues opiniones, problemas, beneficios ni comparaciones que no estén escritos en el contexto.
- No muestres teléfonos ni datos personales.
- Si el contexto no alcanza para responder con seguridad, aclaralo.
- No uses información externa.

Instrucciones según el tipo de recuperación:
{instrucciones_especificas}
"""

    user_prompt = f"""
Consulta del usuario:
{consulta_usuario}

Contexto recuperado:
{contexto}

Redactá la respuesta final usando únicamente el contexto recuperado.
"""

    respuesta = consultar_llm_local(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        max_new_tokens=320
    )

    return respuesta

In [55]:
consultas_respuesta_final = [
    "¿Qué opinan los usuarios de esta cafetera?",
    "¿Cuáles son las licuadoras de menos de 400 dólares con stock?",
    "¿Qué productos están relacionados con la categoría Cocina?"
]

for consulta in consultas_respuesta_final:
    print("=" * 100)
    print("Consulta:", consulta)

    resultado_pipeline = pipeline_recuperacion(
        consulta,
        top_k=10
    )

    respuesta_final = generar_respuesta_final(
        consulta,
        resultado_pipeline
    )

    print("Intención:", resultado_pipeline["intencion"])
    print("Fuente:", resultado_pipeline["fuente"])
    print("\nRespuesta final:")
    print(respuesta_final)
    print()

Consulta: ¿Qué opinan los usuarios de esta cafetera?
Intención: vectorial
Fuente: ChromaDB + BM25 + ReRank

Respuesta final:
Según las reseñas recuperadas, los usuarios tienen opiniones variadas sobre la cafetera. Algunos valoran su relación calidad-precio, durabilidad y versatilidad, otorgándole puntuaciones entre 4 y 5 estrellas. Otros señalan que es inestable, caro y tiene problemas de sobrecalentamiento, otorgándole puntuaciones entre 2 y 3 estrellas. Una reseña menciona que cumple con las expectativas y es muy eficiente, mientras que otra indica que es muy seco y consume mucho energía.

Consulta: ¿Cuáles son las licuadoras de menos de 400 dólares con stock?
Intención: tabular
Fuente: Pandas

Respuesta final:
Se encontraron 3 productos que cumplen con la consulta:
- Licuadora (TechHome): precio USD 283.63, stock 108, voltaje 12V, garantía 36 meses.
- Plus Licuadora Pro (TechHome): precio USD 329.07, stock 97, voltaje 220V, garantía 18 meses.
- Compacto Licuadora (ChefMaster): preci

## Memoria conversacional

In [56]:
historial_conversacion = []


def formatear_historial_conversacion(historial, max_turnos=3):
    # Toma los últimos turnos de la conversación y los deja en texto.
    # Se limita la cantidad para no llenar demasiado el prompt del modelo.

    if len(historial) == 0:
        return "Sin historial previo."

    ultimos_turnos = historial[-max_turnos:]

    partes = []

    for turno in ultimos_turnos:
        partes.append(
            f"Usuario: {turno['consulta']}\n"
            f"Asistente: {turno['respuesta']}"
        )

    return "\n\n".join(partes)


def asistente_rag(consulta_usuario, top_k=10):
    # Ejecuta el sistema completo:
    # 1. clasifica la intención
    # 2. recupera información desde la fuente correspondiente
    # 3. genera la respuesta final
    # 4. guarda el turno en memoria

    resultado_pipeline = pipeline_recuperacion(
        consulta_usuario,
        top_k=top_k
    )

    respuesta = generar_respuesta_final(
        consulta_usuario,
        resultado_pipeline
    )

    historial_conversacion.append({
        "consulta": consulta_usuario,
        "intencion": resultado_pipeline["intencion"],
        "fuente": resultado_pipeline["fuente"],
        "respuesta": respuesta
    })

    return {
        "respuesta": respuesta,
        "resultado_pipeline": resultado_pipeline
    }

In [57]:
consultas_conversacion = [
    "¿Qué opinan los usuarios de esta cafetera?",
    "¿Cuáles son las licuadoras de menos de 400 dólares con stock?",
    "¿Qué productos están relacionados con la categoría Cocina?"
]

for consulta in consultas_conversacion:
    salida = asistente_rag(
        consulta,
        top_k=10
    )

    print("=" * 100)
    print("Usuario:", consulta)
    print("Intención:", salida["resultado_pipeline"]["intencion"])
    print("Fuente:", salida["resultado_pipeline"]["fuente"])
    print("\nAsistente:")
    print(salida["respuesta"])
    print()

Usuario: ¿Qué opinan los usuarios de esta cafetera?
Intención: vectorial
Fuente: ChromaDB + BM25 + ReRank

Asistente:
Según las reseñas recuperadas, los usuarios tienen opiniones variadas sobre la cafetera. Algunos valoran su relación calidad-precio, durabilidad y versatilidad, otorgándole puntuaciones entre 4 y 5 estrellas. Otros señalan que es inestable, caro y tiene problemas de sobrecalentamiento, otorgándole puntuaciones entre 2 y 3 estrellas. Una reseña menciona que cumple con las expectativas y es muy eficiente, mientras que otra indica que es muy seco y consume mucho energía.

Usuario: ¿Cuáles son las licuadoras de menos de 400 dólares con stock?
Intención: tabular
Fuente: Pandas

Asistente:
Se encontraron 3 productos que cumplen con la consulta:
- Licuadora (TechHome): precio USD 283.63, stock 108, voltaje 12V, garantía 36 meses.
- Plus Licuadora Pro (TechHome): precio USD 329.07, stock 97, voltaje 220V, garantía 18 meses.
- Compacto Licuadora (ChefMaster): precio USD 259.42, 

In [58]:
print(formatear_historial_conversacion(historial_conversacion))

Usuario: ¿Qué opinan los usuarios de esta cafetera?
Asistente: Según las reseñas recuperadas, los usuarios tienen opiniones variadas sobre la cafetera. Algunos valoran su relación calidad-precio, durabilidad y versatilidad, otorgándole puntuaciones entre 4 y 5 estrellas. Otros señalan que es inestable, caro y tiene problemas de sobrecalentamiento, otorgándole puntuaciones entre 2 y 3 estrellas. Una reseña menciona que cumple con las expectativas y es muy eficiente, mientras que otra indica que es muy seco y consume mucho energía.

Usuario: ¿Cuáles son las licuadoras de menos de 400 dólares con stock?
Asistente: Se encontraron 3 productos que cumplen con la consulta:
- Licuadora (TechHome): precio USD 283.63, stock 108, voltaje 12V, garantía 36 meses.
- Plus Licuadora Pro (TechHome): precio USD 329.07, stock 97, voltaje 220V, garantía 18 meses.
- Compacto Licuadora (ChefMaster): precio USD 259.42, stock 75, voltaje 220V, garantía 24 meses.

Usuario: ¿Qué productos están relacionados con